In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error as mse

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.functional import mse_loss
import torch.optim.lr_scheduler as lr_scheduler


In [ ]:
# Force garbage collection
import gc
gc.collect()

# Clear the PyTorch cache
torch.cuda.empty_cache()

# training_imputed_full_days

In [ ]:
import numpy as np
import torch
import pandas as pd

In [ ]:
### READ DATA ###
df = pd.read_csv('training_474.csv')
df['time'] = pd.to_datetime(df['time'], format='ISO8601')
df.set_index('time', inplace=True)
print(df.shape)
df.head()

In [ ]:
df_wd = df[['Easterly', 'Northerly']]
df_wd

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert the entire dataset to a PyTorch tensor
X_tensor = torch.tensor(df_wd.values, dtype=torch.float32, device=device)

X_tensor_reshape = X_tensor.reshape(474, 1440, 2)

X_tensor_reshape = X_tensor_reshape.permute(0, 2, 1)  # Resulting shape: (N, D, ts)

In [ ]:
np.savez('training_imputed_full_days.npz', data=X_tensor_reshape.detach().cpu().numpy())

## connect two days

In [ ]:
### READ DATA ###
df = pd.read_csv('training_imputed.csv')
df['time'] = pd.to_datetime(df['time'], format='ISO8601')
df.set_index('time', inplace=True)
print(df.shape)
df.head()

In [ ]:
df_wd = df[['Easterly', 'Northerly']]
df_wd

In [ ]:
# Set all values for June 8-11 (2001), June 11-12 (2005), June 16-17 (2011), June 21 (2018) to NAN
# these days have missing minutes which are not be imputed

df_wd.loc[df.index.normalize().isin([
    pd.Timestamp('2001-06-08'),
    pd.Timestamp('2002-06-09'),
    pd.Timestamp('2002-06-10'),
    pd.Timestamp('2002-06-11'),
    pd.Timestamp('2005-06-11'),
    pd.Timestamp('2005-06-12'),
    pd.Timestamp('2011-06-16'),
    pd.Timestamp('2011-06-17'),
    pd.Timestamp('2018-06-21')
])] = np.nan

In [ ]:
total_nans = df_wd.isna().sum().sum()
print(f"Total NaN values: {total_nans}")

In [ ]:
df_wd.loc['2001-06-08']

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert the entire dataset to a PyTorch tensor
X_tensor = torch.tensor(df_wd.values, dtype=torch.float32, device=device)

X_tensor_reshape = X_tensor.reshape(483, 1440, 2)

X_tensor_reshape = X_tensor_reshape.permute(0, 2, 1)  # Resulting shape: (N, D, ts)
X_tensor_reshape.shape

In [ ]:
# Count NaN values
nan_count = torch.isnan(X_tensor_reshape).sum().item()
print(nan_count)

In [ ]:
data_reshaped = X_tensor_reshape.reshape(23, 21, 2, 1440)
# Step 2: Drop the last day of each year to get 20 days/year
data1 = data_reshaped[:, :20, :, :]  # Shape: (23, 20, 2, 1440)
data2 = data_reshaped[:, 1:, :, :]

In [ ]:
data_2days=torch.concat((data1, data2), dim=3)
data_2days = data_2days.reshape(23*20, 2, 1440*2)

# Count NaN values
nan_count = torch.isnan(data_2days).sum().item()
print(nan_count)
# np.savez('training_impouted_full_2days.npz', data=data_2days)

In [ ]:
rows_with_nans = torch.isnan(data_2days).any(dim=1).any(dim=1)
data_2days_clean = data_2days[~rows_with_nans]  # Selects only rows without NaNs
print("Original shape:", data_2days.shape)
print("Clean shape:", data_2days_clean.shape)  

# Count NaN values
nan_count = torch.isnan(data_2days_clean).sum().item()
print(nan_count)

In [ ]:
np.savez('training_imputed_full_2days.npz', data=data_2days_clean.cpu().numpy())

## visualization

In [ ]:
import numpy as np
days = np.load('training_imputed_full_days.npz')['data']
days.shape

In [ ]:
import matplotlib.pyplot as plt

# Time axis (e.g., 0 to 1439 minutes)
time = np.arange(1440)

plt.figure(figsize=(10, 5))

# Plot both channels
plt.plot(time, days[0, 0], label='Easterly')
plt.plot(time, days[0, 1], label='Northerly')

plt.title("Wind Speed")
plt.xlabel("Time Steps")
plt.ylabel("Value")
# plt.legend()
plt.show()


# Weather_State_10min

In [ ]:
import numpy as np
import pandas as pd
import torch

In [ ]:
### READ DATA ###
df = pd.read_csv('training_weather_state_10min.csv')
df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%dT%H:%M:%SZ')
df.set_index('time', inplace=True)
print(df.shape)
df.head()

In [ ]:
df_ws = df[['Easterly', 'Northerly', 'weather_state']]
df_ws

In [ ]:
### READ DATA for non weather state###
df0 = pd.read_csv('training_imputed_full_days.csv')
df0['time'] = pd.to_datetime(df0['time'], format='ISO8601')
df0.set_index('time', inplace=True)
print(df0.shape)
df0.head()

In [ ]:
# Filter data1 to only keep rows with index in data2['time']
filtered_data1 = df_ws.loc[df_ws.index.intersection(df0.index)]
df_ws = filtered_data1

In [ ]:
# Clean and convert
def weather_to_int(s):
    # Remove parentheses and spaces
    s = s.replace('(', '').replace(')', '').replace(' ', '')
    # Split into components and convert '+' to '1', '-' to '0'
    binary_str = ''.join(['1' if x == '+' else '0' for x in s.split(',')])
    return int(binary_str, 2)  # Convert binary string to integer


In [ ]:
weather_to_int("(+, -, +, +)")

In [ ]:

df_ws['weather_state'] = df_ws['weather_state'].apply(weather_to_int)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert the entire dataset to a PyTorch tensor
X_tensor = torch.tensor(df_ws.values, dtype=torch.float32, device=device)

X_tensor_reshape = X_tensor.reshape(474, 1440, 3)

X_tensor_reshape = X_tensor_reshape.permute(0, 2, 1)  # Resulting shape: (N, D, ts)

np.savez('training_imputed_full_days_ws.npz', data=X_tensor_reshape.detach().cpu().numpy())

## connect two days

In [ ]:
### READ DATA ###
df = pd.read_csv('training_weather_state_10min.csv')
df['time'] = pd.to_datetime(df['time'], format='%Y-%m-%dT%H:%M:%SZ')
df.set_index('time', inplace=True)
print(df.shape)
df.head()

In [ ]:
df_ws = df[['Easterly', 'Northerly', 'weather_state']]
df_ws['weather_state'] = df_ws['weather_state'].apply(weather_to_int)
df_ws

In [ ]:
# Set all values for June 8-11 (2001), June 11-12 (2005), June 16-17 (2011), June 21 (2018) to NAN
# these days have missing minutes which are not be imputed

df_ws.loc[df.index.normalize().isin([
    pd.Timestamp('2001-06-08'),
    pd.Timestamp('2002-06-09'),
    pd.Timestamp('2002-06-10'),
    pd.Timestamp('2002-06-11'),
    pd.Timestamp('2005-06-11'),
    pd.Timestamp('2005-06-12'),
    pd.Timestamp('2011-06-16'),
    pd.Timestamp('2011-06-17'),
    pd.Timestamp('2018-06-21')
])] = np.nan

In [ ]:
total_nans = df_ws.isna().sum().sum()
print(f"Total NaN values: {total_nans}")

In [ ]:
df_ws.loc['2001-06-08']

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert the entire dataset to a PyTorch tensor
X_tensor = torch.tensor(df_ws.values, dtype=torch.float32, device=device)

X_tensor_reshape = X_tensor.reshape(483, 1440, 3)

X_tensor_reshape = X_tensor_reshape.permute(0, 2, 1)  # Resulting shape: (N, D, ts)
X_tensor_reshape.shape

In [ ]:
# Count NaN values
nan_count = torch.isnan(X_tensor_reshape).sum().item()
print(nan_count)

In [ ]:
data_reshaped = X_tensor_reshape.reshape(23, 21, 3, 1440)
# Step 2: Drop the last day of each year to get 20 days/year
data1 = data_reshaped[:, :20, :, :]  # Shape: (23, 20, 3, 1440)
data2 = data_reshaped[:, 1:, :, :]

data_2days=torch.concat((data1, data2), dim=3)
data_2days = data_2days.reshape(23*20, 3, 1440*2)

# Count NaN values
nan_count = torch.isnan(data_2days).sum().item()
print(nan_count)

rows_with_nans = torch.isnan(data_2days).any(dim=1).any(dim=1)
data_2days_clean = data_2days[~rows_with_nans]  # Selects only rows without NaNs
print("Original shape:", data_2days.shape)
print("Clean shape:", data_2days_clean.shape)  

# Count NaN values
nan_count = torch.isnan(data_2days_clean).sum().item()
print(nan_count)

In [ ]:
np.savez('training_imputed_full_2days_ws.npz', data=data_2days_clean.cpu().numpy())